<img src="https://fedesarrollo.org.co/logo-Fede.png" alt="Fedesarrollo" style="float: right; width: 300px; height: auto;">
<img src="https://upload.wikimedia.org/wikipedia/commons/f/f9/Logo_BID_Espa%C3%B1ol.png" alt="Banco Interamericano de Desarrollo" style="float: right; width: 220px; height: auto;">

# Resiliencia de los productos priorizados

Editor: Juan Diego Heredia Niño - jheredia@fedesarrollo.org.co

Septiembre 2026

### Importar librerías

La siguiente celda marca el inicio de un nuevo bloque de trabajo independiente:

**Librerías utilizadas:**
- `pandas`: para manipulación y análisis de datos estructurados.

In [1]:
import pandas as pd

### Configuración de rutas

Carga las rutas del proyecto y las opciones globales desde `config.py`, instalado como
paquete editable en la raíz del proyecto.

In [2]:
from config import raw, temp, tables

### Parámetros

Define el año que se excluye del promedio de concentración y las hojas del archivo de
estrategias de resiliencia declaradas por cada jurisdicción.

In [3]:
anio_excluido = 2013

hojas_estrategias = ['USA', 'EU', 'Canada', 'APEC', 'UK', 'Mexico', 'Ad Hoc']

### Lectura de datos

Carga la cartera con tamaño de mercado, el índice de concentración promedio por producto y el
archivo de estrategias de resiliencia.

In [4]:
df_products = pd.read_parquet(temp / 'df_products_mercado.parquet')

df_hhi = pd.read_parquet(temp / 'df_hhi.parquet').query('Año != @anio_excluido').groupby('HS6')['HHI'].mean().reset_index().rename(columns={'HHI': 'HHI promedio'})

dfs_estrategias = pd.read_excel(raw / 'resiliencia' / 'Estrategias de resiliencia.xlsx', sheet_name=None)

carpeta_resiliencia = tables / 'resiliencia'
carpeta_resiliencia.mkdir(parents=True, exist_ok=True)

### Resiliencia por concentración de mercado

Marca como resiliente al producto cuyo índice de concentración promedio alcanza la mediana de
la cartera.

In [5]:
df_hhi = df_products.merge(df_hhi, on='HS6', how='left').drop(columns=['Fuente','Categoría']).drop_duplicates().sort_values(['Margen','Potencial Exportador Restante','HHI promedio'], ascending=[True, False, False]).reset_index(drop=True)
df_hhi['Resiliencia'] = (df_hhi['HHI promedio'] >= df_hhi['HHI promedio'].median()).astype(int)

df_hhi.to_excel(carpeta_resiliencia / 'df_resiliencia.xlsx', index=False)

### Resiliencia por necesidad declarada

Consolida las estrategias de resiliencia declaradas por las siete jurisdicciones y marca los
productos que aparecen en al menos una de ellas.

El notebook original enumeraba las siete hojas una por una; aquí se recorren con la lista de
parámetros.

In [6]:
df_estrategias = pd.concat(
    [dfs_estrategias[hoja].groupby('HS-2012')['Producto / Material'].count().reset_index() for hoja in hojas_estrategias],
    ignore_index=True
).rename(columns={'HS-2012': 'HS6', 'Producto / Material': 'Cantidad'}).groupby('HS6')['Cantidad'].sum().reset_index()
df_estrategias['HS6'] = df_estrategias['HS6'].astype(str).str.zfill(6)
df_estrategias['Resiliencia ND'] = (df_estrategias['Cantidad'] > 0).astype(int)

df_products.merge(df_estrategias, on='HS6', how='left').drop(columns=['Fuente','Categoría']).drop_duplicates().sort_values(['Margen','Potencial Exportador Restante','Cantidad'], ascending=[True, False, False]).reset_index(drop=True).fillna(0).to_excel(carpeta_resiliencia / 'df_resiliencia_nec_declarada.xlsx', index=False)